# Notebook 05 — UMAP and t-SNE Visualization

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence  
**Notebook:** 05 of 12  
**Time estimate:** 75 minutes

> UMAP and t-SNE are the two standard tools for projecting high-dimensional single-cell
> data to 2D for visualization.
> Both can lie: inter-cluster distances are not interpretable.
> Understanding what they preserve — and what they distort — is essential for correct
> interpretation of the plots you will produce daily.

---
## Step 1 — Motivation

After PCA we have 50 dimensions. Humans can interpret 2. UMAP and t-SNE are
non-linear embeddings that preserve local neighbourhood structure — cells that
are similar in high-dimensional PCA space are placed close together in 2D.

---
## Step 2 — Intuition

**t-SNE:** Build a probability distribution over pairs of cells in high-dimensional
space (P: similar = high probability). Build a second distribution in 2D (Q).
Minimize KL(P||Q) by gradient descent — moving 2D points so Q approximates P.
Result: clusters are tight; distances between clusters are meaningless.

**UMAP:** Similar in spirit but faster. Constructs a fuzzy topological graph in
high-D space using kNN, then optimizes a low-D layout that preserves that graph
structure. UMAP preserves more global structure than t-SNE and is ~10–100× faster.

**Critical caveat for both:** The 2D plot is for *visualization only*. Cluster
sizes and inter-cluster distances should not be interpreted quantitatively.
Clustering must be done in PCA space (NB06), not in the UMAP/t-SNE embedding.

---
## Step 3 — Biological Background

**UMAP in the scRNA-seq workflow:**  
The standard scanpy workflow runs UMAP *after* computing a kNN graph in PCA space.
UMAP then lays out that graph in 2D. This means UMAP is a *visualization* of the
graph, not the basis for clustering. The graph (from NB06) is used for Leiden
clustering; UMAP only displays the result.

**Key parameters:**
- `n_neighbors` (UMAP/t-SNE): controls local vs. global structure. Low n_neighbors
  → tight local clusters, poor global structure. High n_neighbors → smoother,
  better global structure. Default: 15.
- `min_dist` (UMAP): minimum distance between points in 2D. Low → tighter clusters.
  Default: 0.1.
- `perplexity` (t-SNE): analogous to n_neighbors. Default: 30.

---
## Step 4 — Mathematical Explanation

**t-SNE high-dimensional similarities:**
$$p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}$$

Symmetrize: $p_{ij} = (p_{j|i} + p_{i|j}) / 2n$

**t-SNE low-dimensional similarities (Student-t, heavy tails):**
$$q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l}(1 + \|y_k - y_l\|^2)^{-1}}$$

**Objective:** minimize $\text{KL}(P \| Q) = \sum_{ij} p_{ij} \log \frac{p_{ij}}{q_{ij}}$

The heavy-tailed $q_{ij}$ (Student-t vs. Gaussian) prevents the "crowding problem" —
dissimilar points are pushed farther apart in 2D than in high-D, creating visible
cluster separation.

**UMAP** uses fuzzy set membership instead of Gaussian probabilities, with a
cross-entropy loss rather than KL divergence — details in McInnes 2018 (papers.md).

In [ ]:
# Step 6 — UMAP via umap-learn library (Tier 2: tool usage, not from-scratch)
# t-SNE via sklearn. PCA from scratch as established in NB04.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# ---- Generate structured single-cell data ----
N_CELLS = 400
N_GENES = 500
cell_types = rng.choice(['T_cell', 'B_cell', 'Monocyte', 'NK_cell'],
                          N_CELLS, p=[0.3, 0.3, 0.25, 0.15])

ct_colors = {'T_cell': 'steelblue', 'B_cell': 'tomato',
              'Monocyte': 'forestgreen', 'NK_cell': 'purple'}

# Simulate log-normalized expression with 4 programs
programs = {
    'T_cell':   (range(0, 80),   2.5),
    'B_cell':   (range(80, 160), 2.0),
    'Monocyte': (range(160, 240), 3.0),
    'NK_cell':  (range(240, 300), 2.8),
}
X = rng.normal(0, 0.3, (N_CELLS, N_GENES))
for i, ct in enumerate(cell_types):
    idx, level = programs[ct]
    X[i, list(idx)] += level + rng.normal(0, 0.5, len(list(idx)))
X = np.clip(X, 0, None)

# ---- PCA (from scratch, NB04) ----
def pca(X, n_components=50):
    Xc = X - X.mean(axis=0)
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    return (U * s)[:, :n_components]

Z_pca = pca(X, n_components=30)
print(f'PCA: {Z_pca.shape}')

# ---- UMAP ----
try:
    import umap
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                         random_state=42, n_jobs=1)
    Z_umap = reducer.fit_transform(Z_pca)
    print(f'UMAP: {Z_umap.shape}')
except ImportError:
    print('umap-learn not installed. Install with: pip install umap-learn')
    # Fallback: use PCA first 2 components as placeholder
    Z_umap = Z_pca[:, :2]
    print('Using PCA2 as placeholder for UMAP')

# ---- t-SNE ----
try:
    from sklearn.manifold import TSNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=1)
    Z_tsne = tsne.fit_transform(Z_pca)
    print(f't-SNE: {Z_tsne.shape}')
except ImportError:
    print('sklearn not installed.')
    Z_tsne = Z_pca[:, :2]
    print('Using PCA2 as placeholder for t-SNE')

In [ ]:
# Step 7 — Visualization: UMAP vs. t-SNE and parameter sensitivity
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

def scatter_2d(ax, Z, cell_types, ct_colors, title):
    for ct, col in ct_colors.items():
        mask = np.array(cell_types) == ct
        ax.scatter(Z[mask, 0], Z[mask, 1], c=col, s=10, alpha=0.7, label=ct)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, markerscale=2)
    ax.set_xlabel('Dim 1')
    ax.set_ylabel('Dim 2')

# Panel A: PCA2
scatter_2d(axes[0, 0], Z_pca, cell_types, ct_colors, 'A. PCA (PC1 vs PC2)')

# Panel B: UMAP
scatter_2d(axes[0, 1], Z_umap, cell_types, ct_colors, 'B. UMAP (n_neighbors=15)')

# Panel C: t-SNE
scatter_2d(axes[0, 2], Z_tsne, cell_types, ct_colors, 'C. t-SNE (perplexity=30)')

# Panel D-F: UMAP with varying n_neighbors (shows parameter sensitivity)
for j, nn in enumerate([5, 30, 80]):
    ax = axes[1, j]
    try:
        import umap
        r = umap.UMAP(n_neighbors=nn, min_dist=0.1, n_components=2,
                       random_state=42, n_jobs=1)
        Z_u = r.fit_transform(Z_pca)
    except ImportError:
        Z_u = Z_pca[:, :2] + rng.normal(0, nn * 0.1, Z_pca[:, :2].shape)
    scatter_2d(ax, Z_u, cell_types, ct_colors,
                f'D{j+1}. UMAP n_neighbors={nn}')

plt.suptitle('Module 10 NB05: UMAP and t-SNE Visualization', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scrna_umap_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Run UMAP with `n_neighbors=5` and `n_neighbors=50`. Describe how the embedding
   changes and explain why.
2. A colleague says "the monocyte cluster is twice as large as the T cell cluster on
   the UMAP, so there must be twice as many monocytes." Is this correct? Explain.
3. Run UMAP directly on the raw HVG expression matrix (skip PCA). How does the
   result compare to UMAP-on-PCA? What does this tell you about the PCA step?
4. What is the difference between UMAP and t-SNE in terms of preserved structure?
   When might you prefer t-SNE?

---
## Step 10 — Quiz

1. UMAP and t-SNE both preserve _____ structure but distort _____ structure.
2. Why is the kNN graph built in PCA space, not UMAP space?
3. What is the perplexity parameter in t-SNE and what does changing it affect?
4. Can you use UMAP coordinates to compute distances between clusters? Why or why not?
5. In scanpy, UMAP is run with `sc.tl.umap(adata)`. What prerequisite step must be
   run first?

---
## Step 12 — Reflection

> *[In one paragraph: explain the most important limitation of UMAP/t-SNE plots,
> and describe an alternative way to measure distances between cell populations
> that does not rely on the 2D embedding.]*

---
*Next: `06_clustering_leiden_louvain.ipynb`*